# Import

In [1]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [2]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Dependency
* DGIDB_hypergraph
* DDBC

# Prep

## Loading variables

In [3]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [4]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [5]:
# Loading result graph and communities
with open(f"{DISEASE_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{DISEASE_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{DISEASE_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [6]:
for c in communities_selected:
    print(len(c))

863
1000
951
1152
979
905
859
548
417
336
300
127


## Helpful functions (big object, drop NAN)

In [7]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [8]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [9]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['4820', '10724', '6397', '7163', '169200', '57720', '55754', '54978', '157567', '5954', '9019', '11163', '23635', '91404', '9980', '57222', '64423', '3995', '9522', '754', '113419', '64327', '126321', '50717', '9991', '1429', '55186', '56987', '8763', '64776', '9747', '8543', '54629', '23157', '120', '23174', '55884', '440026', '57460', '6509', '84272', '4774', '64215', '127262', '9145', '387263', '55858', '9910', '23673', '6651', '79573', '54708', '91966', '57798', '3338', '79627', '113263', '23731', '4154', '54664', '9202', '23176', '65084', '162427', '57035', '26225', '58497', '55317', '221035', '23199', '51136', '9143', '79180', '54788', '79666', '54842', '55652', '80212', '10807', '55325', '1266', '5891', '8780', '6253', '10472', '26260', '55857', '55103', '55005', '23066', '83941', '90338', '10857', '23613', '124565', '25829', '1362', '55002', '23061', '88455', '162239', '55900', '51315', '4925', '9920', '64771', '11244', '7873', '140890', '23200', '81572', '54557', '51108', '7

In [10]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['28', '48', '100', '104', '120', '141', '166', '175', '178', '203', '205', '226', '267', '271', '286', '287', '310', '323', '333', '353', '373', '402', '421', '427', '440', '444', '473', '475', '476', '483', '520', '550', '576', '577', '586', '631', '636', '665', '676', '734', '750', '754', '757', '770', '819', '831', '833', '987', '989', '1038', '1054', '1102', '1119', '1120', '1130', '1152', '1153', '1174', '1176', '1181', '1182', '1192', '1198', '1200', '1201', '1203', '1266', '1362', '1371', '1389', '1400', '1406', '1411', '1421', '1429', '1506', '1534', '1603', '1611', '1627', '1654', '1656', '1657', '1731', '1762', '1773', '1775', '1777', '1797', '1801', '1829', '1891', '1917', '1939', '1951', '1952', '1983', '2013', '2027', '2038', '2039', '2054', '2070', '2121', '2135', '2137', '2139', '2171', '2195', '2218', '2235', '2281', '2286', '2310', '2504', '2509', '2519', '2530', '2531', '2582', '2583', '2584', '2585', '2589', '2590', '2592', '2597', '2629', '2630', '2632', '2647', '

## NCBI to HGNC

In [11]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [12]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [13]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [14]:
print(len(COMMUNITIES_HGNC))

12


In [15]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 1 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 4 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries

Total dropped across all communities: 6
Community 0: dropped 4 NaN entries
Community 1: dropped 2 NaN entries
Community 2: dropped 4 NaN entries
Community 3: dropped 10 NaN entries
Community 4: dropped 1 NaN entries
Community 5: dropped 2 NaN entries
Community 6: dropped 7 NaN entries
Community 7: dropped 3 NaN entries
Community 8: dropped 4 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 1 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Commun

In [16]:
with open(f"{DISEASE_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{DISEASE_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [17]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

12
16


# Categoization Prep

### GO-slim

In [18]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [19]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [20]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [21]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [22]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [23]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [24]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [25]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [26]:
TERM_SCORE_CAP = 1e-5
PERCENTAGE = 0.1

In [27]:
def enrichment(communities,
               term_score_cap,
               percentage, 
               db,
               term_to_category):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=db,
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["Category"] = filtered["Term"].apply(lambda term: term_to_category(term))

        # Get empty count
        empty_count = (filtered["Category"].apply(len) == 0).sum()
        
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Adjusted P-value'], ascending=True)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "genes_involved": involved,
            "n_involved": len(involved),
            "n_not_involved": len(not_involved)
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

### GO

In [28]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["id"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["id"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "n_involved": len(involved),
            "n_not_involved": len(not_involved),
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

In [29]:
term = "Nuclear Pore Organization (GO:0006999)"
print(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))

{'GO:0009987'}


In [30]:
go_important_terms, go_community_coverage = enrichment(COMMUNITIES_HGNC,
                                                       TERM_SCORE_CAP,
                                                       PERCENTAGE,
                                                       ['GO_Biological_Process_2023',
                                                        'GO_Molecular_Function_2023',
                                                        'GO_Cellular_Component_2023'],
                                                       lambda term: [go[id].name for id in list(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))])

Size of community: 1000
Number of filtered terms: 84
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_77704\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
1689,1,RNA Binding (GO:0003723),357/1411,1.671873e-163,[binding]
0,1,"mRNA Splicing, Via Spliceosome (GO:0000398)",97/211,4.641563e-66,[cellular process]
1,1,Mitochondrial Translation (GO:0032543),70/98,4.641563e-66,[cellular process]
2,1,mRNA Processing (GO:0006397),97/214,9.280206e-66,[cellular process]
3,1,"RNA Splicing, Via Transesterification Reactions With Bulged Adenosine As Nucleophile (GO:0000377)",90/180,9.304640e-66,[cellular process]
4,1,Mitochondrial Gene Expression (GO:0140053),70/103,5.414255e-64,[cellular process]
2072,1,Nuclear Lumen (GO:0031981),163/780,9.457570e-56,[cellular anatomical structure]
2073,1,Nucleolus (GO:0005730),161/771,2.893673e-55,[cellular anatomical structure]
5,1,Translation (GO:0006412),83/234,5.960891e-46,[cellular process]
6,1,Ribosome Biogenesis (GO:0042254),69/155,8.243541e-46,[cellular process]


Size of community: 951
Number of filtered terms: 18
Number of unmapped terms: 1


,Community Index,Term,Overlap,Adjusted P-value,Category
2817,2,Collagen-Containing Extracellular Matrix (GO:0062023),110/373,1.120578e-54,[]
0,2,Extracellular Matrix Organization (GO:0030198),46/176,1.768867e-18,[cellular process]
1,2,Supramolecular Fiber Organization (GO:0097435),49/316,2.874933e-10,[cellular process]
2818,2,Endoplasmic Reticulum Lumen (GO:0005788),44/284,4.193806e-10,[cellular anatomical structure]
2819,2,Cell-Substrate Junction (GO:0030055),51/395,5.911239e-09,[cellular anatomical structure]
2820,2,Focal Adhesion (GO:0005925),50/387,6.641794e-09,[cellular anatomical structure]
2,2,Collagen Fibril Organization (GO:0030199),16/42,2.501619e-08,[cellular process]
2821,2,Basement Membrane (GO:0005604),15/46,6.627341e-08,[cellular anatomical structure]
2398,2,Calcium Ion Binding (GO:0005509),46/346,8.852986e-08,[binding]
2399,2,Endopeptidase Inhibitor Activity (GO:0004866),24/112,8.852986e-08,[molecular function regulator activity]


...

Size of community: 300
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
21,10,Olfactory Receptor Activity (GO:0004984),250/362,0.000000e+00,[molecular transducer activity]
0,10,Sensory Perception Of Smell (GO:0007608),148/230,7.105019e-225,[multicellular organismal process]
1,10,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),89/141,2.233247e-129,[response to stimulus]
2,10,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),88/139,3.308015e-128,[response to stimulus]
3,10,Sensory Perception Of Chemical Stimulus (GO:0007606),67/110,5.372131e-95,[multicellular organismal process]


Size of community: 127
Number of filtered terms: 23
Number of unmapped terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
2,11,Cellular Glucuronidation (GO:0052695),7/15,6.332094e-10,[]
3,11,Glucuronate Metabolic Process (GO:0019585),7/15,6.332094e-10,[cellular process]
1166,11,Serotonin Receptor Activity (GO:0099589),8/26,7.082150e-10,[molecular transducer activity]
1167,11,Retinoic Acid Binding (GO:0001972),7/19,1.965383e-09,[binding]
1170,11,G Protein-Coupled Neurotransmitter Receptor Activity (GO:0099528),5/6,2.686843e-09,[molecular transducer activity]
1169,11,G Protein-Coupled Acetylcholine Receptor Activity (GO:0016907),5/6,2.686843e-09,[molecular transducer activity]
1168,11,G Protein-Coupled Serotonin Receptor Activity (GO:0004993),7/21,2.686843e-09,[molecular transducer activity]
4,11,Adenylate Cyclase-Inhibiting G Protein-Coupled Receptor Signaling Pathway (GO:0007193),9/52,8.584016e-09,"[cellular process, biological regulation]"
1172,11,Monocarboxylic Acid Binding (GO:0033293),7/28,1.258576e-08,[binding]
1171,11,G Protein-Coupled Amine Receptor Activity (GO:0008227),7/28,1.258576e-08,[molecular transducer activity]


10 out of 12 communities had significant GO terms.


In [31]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,1,1000,RNA Binding (GO:0003723),357/1411,1.671873e-163,[binding],GO_Molecular_Function_2023,4.365203e-166,0.0,0.0,9.453319,3599.402687,POP5;SLC4A1AP;POP1;RTCA;RRP1;PPAN;FCF1;HNRNPU;...,0.253012
1,1,1000,"mRNA Splicing, Via Spliceosome (GO:0000398)",97/211,4.641563e-66,[cellular process],GO_Biological_Process_2023,3.057248e-69,0.0,0.0,17.795866,2807.491001,GEMIN2;HNRNPU;HNRNPR;CASC3;CWC27;PNN;SNRPD2;SN...,0.459716
2,1,1000,Mitochondrial Translation (GO:0032543),70/98,4.641563e-66,[cellular process],GO_Biological_Process_2023,5.496226e-69,0.0,0.0,51.000000,8015.889794,MRPS17;GFM1;MRPS15;MRPS16;FASTKD2;GFM2;MRPS14;...,0.714286
3,1,1000,mRNA Processing (GO:0006397),97/214,9.280206e-66,[cellular process],GO_Biological_Process_2023,1.648349e-68,0.0,0.0,17.336807,2705.859703,GEMIN2;HNRNPU;HNRNPR;CASC3;CWC27;PNN;SNRPD2;SN...,0.453271
4,1,1000,"RNA Splicing, Via Transesterification Reaction...",90/180,9.304640e-66,[cellular process],GO_Biological_Process_2023,2.203586e-68,0.0,0.0,20.780220,3237.261094,DDX46;HNRNPU;PPWD1;HNRNPR;CASC3;CWC27;PNN;SYNC...,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
871,11,127,Xenobiotic Glucuronidation (GO:0052697),4/6,2.983423e-06,[],GO_Biological_Process_2023,2.302814e-08,0.0,0.0,323.105691,5682.314054,UGT1A9;UGT1A8;UGT1A7;UGT1A6,0.666667
872,11,127,Adenylate Cyclase-Activating Adrenergic Recept...,5/15,3.173516e-06,"[cellular process, biological regulation]",GO_Biological_Process_2023,2.721712e-08,0.0,0.0,81.405738,1418.040708,AKAP13;ADRB3;ADRA2B;ADRA2A;DRD5,0.333333
873,11,127,G Protein-Coupled Acetylcholine Receptor Signa...,5/17,5.884711e-06,"[cellular process, biological regulation]",GO_Biological_Process_2023,5.551614e-08,0.0,0.0,67.831284,1133.229591,CHRM2;CHRM3;CHRM4;CHRM5;HRH4,0.294118
874,11,127,Liver Development (GO:0001889),6/35,7.286795e-06,[developmental process],GO_Biological_Process_2023,8.124214e-08,0.0,0.0,33.931034,553.952362,UGT1A10;ARID5B;UGT1A9;UGT1A8;GFER;UGT1A7,0.171429


In [32]:
go_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,863,[],0,863
1,1,1000,"[AARS1, AATF, ABCE1, ABT1, ACTR6, AIMP1, ALDH1...",566,434
2,2,951,"[ABI3BP, ABLIM1, ADAM19, ADAMTS1, ADAMTS2, ADA...",303,648
3,3,1151,"[ACE2, ACKR1, ACKR4, ACOD1, ADAM28, ADAM32, AD...",496,655
4,4,979,"[ABHD17A, ABHD17B, ABI1, ABI2, ABLIM2, ABLIM3,...",824,155
5,5,905,"[ABRAXAS2, ACTL6A, ACTL6B, ADNP2, AEBP2, AGO2,...",759,146
6,6,855,"[AASS, ABAT, ABCA5, ABCA6, ABCA8, ABCA9, ABCB4...",215,640
7,7,548,"[ABCC8, ADGRL3, ADORA1, ADORA2A, ADORA2B, ALX1...",270,278
8,8,417,"[ANLN, ANP32E, ASF1A, ATAD5, AURKA, AURKB, BAR...",234,183
9,9,335,[],0,335


### KEGG

In [33]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [34]:
kegg_important_terms, kegg_community_coverage = enrichment(communities = COMMUNITIES_HGNC,
                                 term_score_cap = TERM_SCORE_CAP,
                                 percentage = PERCENTAGE,
                                 db = ['KEGG_2021_Human'],
                                 term_to_category = lambda term: get_kegg_level2(name_to_id.get(term.lower())))

Size of community: 1000
Number of filtered terms: 5
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_77704\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,1,Spliceosome,69/150,1.704140e-47,[Transcription]
1,1,Ribosome biogenesis in eukaryotes,43/108,2.197768e-26,[Translation]
2,1,RNA transport,38/186,2.247775e-12,[]
3,1,mRNA surveillance pathway,27/98,3.520122e-12,[Translation]
4,1,Ribosome,33/158,3.054144e-11,[Translation]


Size of community: 1151
Number of filtered terms: 18
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,3,Cytokine-cytokine receptor interaction,144/295,8.433273e-97,[Signaling molecules and interaction]
1,3,Viral protein interaction with cytokine and cytokine receptor,66/100,2.087707e-55,[Signaling molecules and interaction]
2,3,NF-kappa B signaling pathway,35/104,1.954596e-16,[Signal transduction]
3,3,Chemokine signaling pathway,46/192,3.003903e-15,[Immune system]
4,3,Hematopoietic cell lineage,32/99,1.249370e-14,[Immune system]
5,3,Rheumatoid arthritis,29/93,7.406838e-13,[Immune disease]
6,3,Natural killer cell mediated cytotoxicity,33/131,9.421739e-12,[Immune system]
7,3,JAK-STAT signaling pathway,36/162,4.059404e-11,[Signal transduction]
8,3,IL-17 signaling pathway,26/94,2.355107e-10,[Immune system]
9,3,TNF signaling pathway,28/112,4.927772e-10,[Signal transduction]


...

Size of community: 300
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Olfactory transduction,297/440,0.0,[Sensory system]


Size of community: 127
Number of filtered terms: 11
Number of unmapped terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Drug metabolism,18/108,1.440908e-18,[]
2,11,Metabolism of xenobiotics by cytochrome P450,12/76,3.819725e-12,[Xenobiotics biodegradation and metabolism]
4,11,Ascorbate and aldarate metabolism,8/30,4.072743e-10,[Carbohydrate metabolism]
5,11,Bile secretion,11/90,4.072743e-10,[Digestive system]
7,11,Pentose and glucuronate interconversions,7/34,4.231871e-08,[Carbohydrate metabolism]
10,11,Porphyrin and chlorophyll metabolism,7/43,1.611867e-07,[]
11,11,Type I diabetes mellitus,7/43,1.611867e-07,[Endocrine and metabolic disease]
12,11,Retinol metabolism,8/68,1.801264e-07,[Metabolism of cofactors and vitamins]
14,11,Steroid hormone biosynthesis,7/61,1.588271e-06,[Lipid metabolism]
15,11,Allograft rejection,6/38,1.751558e-06,[Immune disease]


9 out of 12 communities had significant GO terms.


In [35]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,1,1000,Spliceosome,69/150,1.704140e-47,[Transcription],KEGG_2021_Human,2.157139e-49,0.0,0.0,17.310618,1939.791253,RBM25;DDX46;HNRNPU;PQBP1;SNRPD2;SNRPD1;SNRNP70...,0.460000
1,1,1000,Ribosome biogenesis in eukaryotes,43/108,2.197768e-26,[Translation],KEGG_2021_Human,5.563969e-28,0.0,0.0,13.089060,821.418001,POP5;RBM28;NXT1;POP1;NVL;RPP30;WDR3;HEATR1;FCF...,0.398148
2,1,1000,RNA transport,38/186,2.247775e-12,[],KEGG_2021_Human,8.535856e-14,0.0,0.0,5.031578,151.409831,POP5;NXT1;RBM8A;POP1;GEMIN2;RPP30;PHAX;NMD3;CA...,0.204301
3,1,1000,mRNA surveillance pathway,27/98,3.520122e-12,[Translation],KEGG_2021_Human,1.782340e-13,0.0,0.0,7.398101,217.176273,DAZAP1;NXT1;RBM8A;CSTF3;CSTF2;CASC3;SMG7;GSPT1...,0.275510
4,1,1000,Ribosome,33/158,3.054144e-11,[Translation],KEGG_2021_Human,1.933002e-12,0.0,0.0,5.153051,138.987808,MRPS17;MRPS15;MRPS16;MRPS14;MRPS11;MRPL19;MRPS...,0.208861
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,11,127,Type I diabetes mellitus,7/43,1.611867e-07,[Endocrine and metabolic disease],KEGG_2021_Human,9.389515e-09,0.0,0.0,32.143287,594.125980,HLA-DRB5;PTPRN2;HLA-B;HLA-DPB1;HLA-C;HLA-DRB3;IL2,0.162791
154,11,127,Retinol metabolism,8/68,1.801264e-07,[Metabolism of cofactors and vitamins],KEGG_2021_Human,1.136720e-08,0.0,0.0,22.199440,406.083998,UGT1A10;UGT1A4;UGT1A9;CYP2C18;UGT1A8;UGT2B7;UG...,0.117647
155,11,127,Steroid hormone biosynthesis,7/61,1.588271e-06,[Lipid metabolism],KEGG_2021_Human,1.156508e-07,0.0,0.0,21.409414,341.965933,UGT1A10;UGT1A4;UGT1A9;UGT1A8;UGT2B7;UGT1A7;UGT1A6,0.114754
156,11,127,Allograft rejection,6/38,1.751558e-06,[Immune disease],KEGG_2021_Human,1.360433e-07,0.0,0.0,30.745351,486.092995,HLA-DRB5;HLA-B;HLA-DPB1;HLA-C;HLA-DRB3;IL2,0.157895


In [36]:
kegg_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,863,[],0,863
1,1,1000,"[AK6, BCAS2, BMS1, BUD31, CASC3, CDC40, CDC5L,...",180,820
2,2,951,[],0,951
3,3,1151,"[ACKR1, ACKR4, AIM2, ANTXR1, ANTXR2, B2M, BCL2...",289,862
4,4,979,"[ABI1, ABI2, ABLIM2, ABLIM3, ACAP1, ACTA2, ACT...",683,296
5,5,905,"[ASPSCR1, ATXN2L, ATXN3, BIRC6, BMI1, CCNA1, C...",166,739
6,6,855,"[ABAT, ABCA5, ABCA6, ABCA8, ABCA9, ABCB4, ABCB...",155,700
7,7,548,"[ADORA1, ADORA2A, ADORA2B, APLNR, AVPR1A, AVPR...",71,477
8,8,417,"[AURKA, BUB1, BUB1B, BUB3, CCNA2, CCNB1, CCNB2...",43,374
9,9,335,[],0,335


### Reactome

In [37]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [38]:
reactome_important_terms, reactome_community_coverage = enrichment(COMMUNITIES_HGNC,
                                      TERM_SCORE_CAP,
                                      PERCENTAGE,
                                      ['Reactome_2022'],
                                      lambda term: reactome_level1.get(term.split(" ")[-1],[]))

Size of community: 1000
Number of filtered terms: 21
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_77704\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,1,Metabolism Of RNA R-HSA-8953854,199/666,2.267918e-99,[Metabolism of RNA]
1,1,Processing Of Capped Intron-Containing Pre-mRNA R-HSA-72203,104/242,2.579276e-68,[Metabolism of RNA]
2,1,mRNA Splicing R-HSA-72172,91/189,3.545194e-65,[Metabolism of RNA]
3,1,mRNA Splicing - Major Pathway R-HSA-72163,87/181,2.849138e-62,[Metabolism of RNA]
4,1,Mitochondrial Translation R-HSA-5368287,64/88,3.970234e-62,[Metabolism of proteins]
5,1,Mitochondrial Translation Elongation R-HSA-5389840,61/82,2.438350e-60,[Metabolism of proteins]
6,1,Mitochondrial Translation Termination R-HSA-5419276,61/82,2.438350e-60,[Metabolism of proteins]
7,1,Mitochondrial Translation Initiation R-HSA-5368286,60/82,1.196400e-58,[Metabolism of proteins]
8,1,Translation R-HSA-72766,96/281,1.223778e-52,[Metabolism of proteins]
9,1,rRNA Modification In Nucleus And Cytosol R-HSA-6790901,41/60,5.853319e-38,[Metabolism of RNA]


Size of community: 951
Number of filtered terms: 9
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Extracellular Matrix Organization R-HSA-1474244,71/291,2.081738e-28,[Extracellular matrix organization]
1,2,Collagen Formation R-HSA-1474290,27/90,1.218211e-12,[Extracellular matrix organization]
2,2,Regulation Of IGF Transport And Uptake By IGFBPs R-HSA-381426,30/123,1.169938e-11,[Metabolism of proteins]
3,2,Post-translational Protein Phosphorylation R-HSA-8957275,27/106,5.023680e-11,[Metabolism of proteins]
4,2,Elastic Fibre Formation R-HSA-1566948,16/39,7.558323e-10,[Extracellular matrix organization]
5,2,Collagen Biosynthesis And Modifying Enzymes R-HSA-1650814,19/67,1.321167e-08,[Extracellular matrix organization]
6,2,Platelet Degranulation R-HSA-114608,23/125,1.423607e-06,[Hemostasis]
7,2,Response To Elevated Platelet Cytosolic Ca2+ R-HSA-76005,23/130,2.674384e-06,[Hemostasis]
8,2,Crosslinking Of Collagen Fibrils R-HSA-2243919,7/10,2.945972e-06,[Extracellular matrix organization]


...

Size of community: 300
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Sensory Perception R-HSA-9709957,292/616,0.0,[Sensory Perception]
1,10,Olfactory Signaling Pathway R-HSA-381753,292/401,0.0,[Sensory Perception]
2,10,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,292/393,0.0,[Sensory Perception]


Size of community: 127
Number of filtered terms: 9
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Biological Oxidations R-HSA-211859,23/218,3.787537e-19,[Metabolism]
1,11,Amine Ligand-Binding Receptors R-HSA-375280,11/40,1.483078e-13,[Signal Transduction]
2,11,Drug ADME R-HSA-9748784,14/95,1.483078e-13,[Drug ADME]
3,11,Phase II - Conjugation Of Compounds R-HSA-156580,12/107,3.805875e-10,[Metabolism]
4,11,Phase I - Functionalization Of Compounds R-HSA-211945,11/104,4.768673e-09,[Metabolism]
6,11,Glucuronidation R-HSA-156588,7/25,8.885952e-09,[Metabolism]
9,11,Paracetamol ADME R-HSA-9753281,7/28,1.508159e-08,[Drug ADME]
12,11,Muscarinic Acetylcholine Receptors R-HSA-390648,4/5,2.397229e-07,[Signal Transduction]
13,11,Aspirin ADME R-HSA-9749641,7/44,3.205275e-07,[Drug ADME]


10 out of 12 communities had significant GO terms.


In [39]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,1,1000,Metabolism Of RNA R-HSA-8953854,199/666,2.267918e-99,[Metabolism of RNA],Reactome_2022,6.535787e-102,0.0,0.0,9.859375,2297.100264,LTV1;POP5;FCF1;HNRNPU;HNRNPR;PHAX;PWP2;RRP9;CC...,0.298799
1,1,1000,Processing Of Capped Intron-Containing Pre-mRN...,104/242,2.579276e-68,[Metabolism of RNA],Reactome_2022,1.486614e-70,0.0,0.0,15.864777,2550.809598,HNRNPU;HNRNPR;CASC3;CWC27;CCAR1;SNRPD2;MAGOH;P...,0.429752
2,1,1000,mRNA Splicing R-HSA-72172,91/189,3.545194e-65,[Metabolism of RNA],Reactome_2022,3.065009e-67,0.0,0.0,19.308974,2957.230168,DDX46;HNRNPU;PPWD1;HNRNPR;CASC3;CWC27;CCAR1;PQ...,0.481481
3,1,1000,mRNA Splicing - Major Pathway R-HSA-72163,87/181,2.849138e-62,[Metabolism of RNA],Reactome_2022,3.284309e-64,0.0,0.0,19.165505,2801.542460,DDX46;HNRNPU;PPWD1;HNRNPR;CASC3;CWC27;CCAR1;PQ...,0.480663
4,1,1000,Mitochondrial Translation R-HSA-5368287,64/88,3.970234e-62,[Metabolism of proteins],Reactome_2022,5.720799e-64,0.0,0.0,54.062678,7872.679487,MRPS17;GFM1;MRPS15;MRPS16;GFM2;MRPS14;MRPS11;M...,0.727273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615,11,127,Phase I - Functionalization Of Compounds R-HSA...,11/104,4.768673e-09,[Metabolism],Reactome_2022,5.901823e-11,0.0,0.0,20.168706,475.037054,CYP2J2;ALDH3A1;NQO2;MAOB;ALDH2;AADAC;FMO1;AHR;...,0.105769
616,11,127,Glucuronidation R-HSA-156588,7/25,8.885952e-09,[Metabolism],Reactome_2022,1.539645e-10,0.0,0.0,64.344907,1453.828075,UGT1A10;UGT1A4;UGT1A9;UGT1A8;UGT1A7;UGT2B7;UGT1A6,0.280000
617,11,127,Paracetamol ADME R-HSA-9753281,7/28,1.508159e-08,[Drug ADME],Reactome_2022,3.733066e-10,0.0,0.0,55.144444,1197.109851,UGT1A10;ABCC1;GSTM1;GSTP1;GSTT1;UGT1A9;UGT1A6,0.250000
618,11,127,Muscarinic Acetylcholine Receptors R-HSA-390648,4/5,2.397229e-07,[Signal Transduction],Reactome_2022,7.713856e-09,0.0,0.0,646.243902,12071.996175,CHRM2;CHRM3;CHRM4;CHRM5,0.800000


In [40]:
reactome_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,863,[],0,863
1,1,1000,"[AARS1, AIMP1, AIMP2, APEH, AURKAIP1, BCAS2, B...",310,690
2,2,951,"[ADAM12, ADAM19, ADAMTS1, ADAMTS2, ADAMTS5, AG...",116,835
3,3,1151,"[ACKR1, ACKR4, ADA2, ADAM8, ADAR, ADGRE1, ADGR...",415,736
4,4,979,"[AAMP, ABHD17A, ABHD17B, ABI1, ABI2, ABLIM2, A...",775,204
5,5,905,"[ABRAXAS2, ACTL6A, ACTL6B, AEBP2, AGO2, AIFM2,...",675,230
6,6,855,"[AASS, ABCB4, ABCC3, ABCD2, ABCD3, ABHD14B, AC...",267,588
7,7,548,"[ABCC8, ADORA1, ADORA2A, ADORA2B, APLNR, AVPR1...",113,435
8,8,417,"[AURKA, AURKB, BARD1, BORA, BUB1, BUB1B, BUB3,...",156,261
9,9,335,[],0,335


# Important Terms df

In [41]:
community_coverage_combined = go_community_coverage.copy()

community_coverage_combined["genes_involved"] = [
    set(a) | set(b) | set(c)
    for a, b, c in zip(go_community_coverage["genes_involved"], kegg_community_coverage["genes_involved"], reactome_community_coverage["genes_involved"])
]
community_coverage_combined["n_involved"] = community_coverage_combined["genes_involved"].apply(len)
community_coverage_combined["n_not_involved"] = community_coverage_combined["n_genes"] - community_coverage_combined["n_involved"]
community_coverage_combined["percentage_involved"] = community_coverage_combined["n_involved"] / community_coverage_combined["n_genes"]

In [42]:
community_coverage_combined

,community,n_genes,genes_involved,n_involved,n_not_involved,percentage_involved
0,0,863,{},0,863,0.000000
1,1,1000,"{TIMM22, PTCD3, HNRNPA1, NOP58, SRSF1, KLHL7, ...",583,417,0.583000
2,2,951,"{PHACTR2, RECK, MYO1B, LTBP2, FAM107A, SPINT1,...",332,619,0.349106
3,3,1151,"{SELE, PTAFR, CLEC7A, ITGAM, CD38, CCL22, CD5L...",591,560,0.513467
4,4,979,"{ARPC4, ITPR1, EPHB6, DUSP16, CPEB1, VAV3, CAL...",929,50,0.948927
5,5,905,"{ZKSCAN8, GOLGB1, TCF3, TLK2, H2BC18, EXOG, US...",845,60,0.933702
6,6,855,"{CDO1, ME1, PCBD1, EPHX1, DAO, CES1, CROT, RBP...",319,536,0.373099
7,7,548,"{CDX2, RIMBP3, SIM2, GPR4, HOXD12, RGS22, NTSR...",289,259,0.527372
8,8,417,"{CKAP2L, FOXM1, PTTG2, ERCC6L, MCM6, ECT2, PRI...",255,162,0.611511
9,9,335,{},0,335,0.000000


In [43]:
comm_to_involved_pct = dict(zip(community_coverage_combined["community"], community_coverage_combined["percentage_involved"]))

with open(DISEASE_FOLDER + "comm_to_involved_pct.json", "w") as f:
    json.dump(comm_to_involved_pct, f, indent=2)


In [44]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")

# for category, keep only the first category and make it a string instead of a list
important_terms["Category"] = important_terms["Category"].apply(lambda x: x[0] if len(x) > 0 else "None")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,1,1000,RNA Binding (GO:0003723),357/1411,1.671873e-163,binding,GO_Molecular_Function_2023,4.365203e-166,0.0,0.0,9.453319,3599.402687,POP5;SLC4A1AP;POP1;RTCA;RRP1;PPAN;FCF1;HNRNPU;...,0.253012
81,1,1000,90S Preribosome (GO:0030686),7/12,3.905668e-06,protein-containing complex,GO_Cellular_Component_2023,4.857796e-07,0.0,0.0,26.780463,389.321274,RSL1D1;NOP14;TBL3;UTP4;HEATR1;SRFBP1;UTP20,0.583333
80,1,1000,Translocation Of Molecules Into Host (GO:0044417),6/7,3.352567e-06,biological process involved in interspecies in...,GO_Biological_Process_2023,1.032170e-07,0.0,0.0,114.682093,1844.825741,DDX39B;THOC1;THOC2;THOC5;THOC7;THOC6,0.857143
79,1,1000,Viral mRNA Export From Host Cell Nucleus (GO:0...,6/7,3.352567e-06,viral process,GO_Biological_Process_2023,1.032170e-07,0.0,0.0,114.682093,1844.825741,DDX39B;THOC1;THOC2;THOC5;THOC7;THOC6,0.857143
78,1,1000,Regulation Of RNA Metabolic Process (GO:0051252),19/91,3.265891e-06,biological regulation,GO_Biological_Process_2023,9.539590e-08,0.0,0.0,5.091630,82.307369,SF3B5;ENY2;SF3B3;FUS;SRSF1;RRP1B;PTBP2;RC3H2;P...,0.208791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
864,11,127,Glucuronosyltransferase Activity (GO:0015020),7/30,1.665478e-08,catalytic activity,GO_Molecular_Function_2023,6.351398e-10,0.0,0.0,50.344203,1066.148045,UGT1A10;UGT1A4;UGT1A9;UGT1A8;UGT1A7;UGT2B7;UGT1A6,0.233333
865,11,127,Adenylate Cyclase-Inhibiting G Protein-Coupled...,5/7,3.852245e-08,cellular process,GO_Biological_Process_2023,1.982287e-10,0.0,0.0,407.192623,9097.334550,CHRM2;CHRM3;CHRM4;CHRM5;HRH4,0.714286
866,11,127,"G Protein-Coupled Receptor Signaling Pathway, ...",8/50,1.512759e-07,cellular process,GO_Biological_Process_2023,9.081745e-10,0.0,0.0,31.742297,660.861435,CHRM2;CHRM3;HTR1E;CHRM4;NPY;CHRM5;HRH4;HTR1B,0.160000
860,11,127,Adenylate Cyclase-Inhibiting G Protein-Coupled...,9/52,8.584016e-09,cellular process,GO_Biological_Process_2023,3.680968e-11,0.0,0.0,35.173433,845.050891,CHRM2;CHRM3;GRM7;HTR1E;CHRM4;CHRM5;HRH4;HTR1B;...,0.173077


In [45]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [46]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [47]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [48]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [49]:
# twr3

In [50]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [51]:
# terms_with_recurrence

In [52]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [53]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [54]:
# terms_with_rec_merged

In [55]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [56]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [57]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [58]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))